In [ ]:
!pip install selenium pandas
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv

BASE_URL = "PASTE_YOUR_LINK_HERE?page="

def get_game_data(page):
    url = BASE_URL + str(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    games = soup.find_all("div", class_="game")  # adjust if needed
    data = []

    for game in games:
        try:
            name = game.find("h2").text.strip()
        except:
            name = ""

        try:
            author = game.find("span", class_="author").text.strip()
        except:
            author = ""

        try:
            description = game.find("p").text.strip()
        except:
            description = ""

        # simple placeholders if not found
        artwork = ""
        code = ""
        license = ""
        likes = ""
        comments = ""

        data.append([
            name, author, artwork, code,
            license, likes, description, comments
        ])

    return data


def main():
    all_data = []

    page = 1
    while len(all_data) < 100:
        print(f"Scraping page {page}")
        page_data = get_game_data(page)

        if not page_data:
            break

        all_data.extend(page_data)
        page += 1

    all_data = all_data[:100]

    with open("games.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow([
            "name", "author", "artwork",
            "code", "license", "likes",
            "description", "comments"
        ])

        writer.writerows(all_data)

    print("Saved 100 games to games.csv")


if __name__ == "__main__":
    main()

# Task 2 : RAG

In [ ]:
import csv
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# read csv
texts = []

with open("games.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        text = row["name"] + " " + row["description"]
        texts.append(text)

# create embeddings
embeddings = model.encode(texts)

# create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("RAG database ready!")

# simple query
def search(query):
    q_embedding = model.encode([query])
    D, I = index.search(np.array(q_embedding), k=3)

    for i in I[0]:
        print("Result:", texts[i])


if __name__ == "__main__":
    while True:
        q = input("Ask something: ")
        search(q)

ModuleNotFoundError: No module named 'faiss'